# Hafta 3 - Analiz: Bir MWh'nin yolculuğu

**Soru:** Türkiye'de elektriğin ne kadarı PTF'den fiyatlanıyor, bu pay mevsime ve yıllara göre nasıl değişiyor ve değişimi ne sürüklüyor?

| Adım | Ne yapıyoruz |
|---|---|
| 1 | İkili anlaşma (İA), GÖP, GİP ve dengeleme hacimlerini tüketime oranlıyoruz (aylık, 2023-2026) |
| 2 | Aynısını saat bazında yapıyoruz |
| 3 | GÖP oranını yenilenebilir payıyla karşılaştırıyoruz (YEKDEM verisi servis boş döndüğü için vekil) |
| 4 | Aylık PTF oynaklığını hesaplıyoruz |
| 5 | İA ve GÖP oranının fiyat seviyesi ve oynaklıkla ilişkisine bakıyoruz (yıl ve ay sabit etkili) |
| 6 | Piyasalar arası fiyat farkları: PTF, GİP, SMF |

**Önemli tanım:** EPİAŞ'ın "piyasa hacimleri" verisi alış ve satışı ayrı sayar (her işlem iki kez). Bu notebook tek taraflı miktarı kullanır: İA = alış bildirimleri, GÖP = eşleşen alış, GİP = eşleşen miktar, DGP = teslim edilen YAL + YAT. Tüketim: gerçek zamanlı tüketim.

In [ ]:
# 0) Ayarlar, stil, veri
import json, warnings, logging
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore"); logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
ISLN = Path("02_islenmis"); OUT = Path("03_grafikler/hafta03"); OUT.mkdir(parents=True, exist_ok=True)
NV, OR, G1, G2, G3 = "#0a1f6b", "#e8590c", "#c3d0ee", "#7f9bd6", "#2f4fb0"
YIL_RENK = {2023: G1, 2024: G2, 2025: G3, 2026: OR}
plt.rcParams.update({"font.family": "DejaVu Sans Mono", "font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": NV, "axes.linewidth": 1.5, "axes.labelcolor": NV, "xtick.color": NV, "ytick.color": NV,
    "axes.grid": True, "grid.color": "#c9d4ee", "grid.linestyle": ":", "axes.axisbelow": True})
def kaydet(fig, ad): fig.savefig(OUT / f"{ad}.png", dpi=200, facecolor="white", bbox_inches="tight"); plt.show()

t = pd.read_csv(ISLN / "ana_tablo_h03_saatlik.csv", parse_dates=["zaman"]).set_index("zaman")
h = pd.read_csv(ISLN / "h03_hacim.csv")
h["zaman"] = pd.to_datetime(h.date, utc=True).dt.tz_convert("Europe/Istanbul").dt.tz_localize(None)
h = h.drop(columns="date").set_index("zaman"); h = h[~h.index.duplicated()]
SON = "2026-10-31 23:00"                       # tam aylar
j = t.join(h).loc[:SON].copy()
j["ren"] = j[["uretim_wind", "uretim_solar", "uretim_river", "uretim_geothermal", "uretim_biomass"]].sum(axis=1)
j["dgp_mwh"] = j.yal_upRegulationDelivered - j.yat_downRegulationDelivered
j["kur"] = (j.ptf / j.ptf_usd).where(j.ptf_usd > 1).groupby(j.index.date).transform("median")
j["gip_usd"], j["smf_usd"] = j.gip / j.kur, j.smf / j.kur
j["yil"], j["saat"] = j.index.year, j.index.hour
# Kontrol: EPİAŞ hacim verisi iki taraflı mı? (iaHacmi / İA alış ≈ 2 olmalı)
print("iaHacmi / İA alış:", round((j.iaHacmi.sum() / j.ia_alis_quantity.sum()), 3),
      "| gopHacmi / GÖP eşleşme:", round(j.gopHacmi.sum() / j.gop_matchedBids.sum(), 3))
print("Aralık:", j.index.min(), "-", j.index.max(), "|", len(j), "saat")

## 1) Bir MWh nerede el değiştiriyor? (tüketime oran, %)

In [ ]:
m = j.resample("MS").sum(numeric_only=True)
r = pd.DataFrame(index=m.index)
T = m.tuketim_mwh
r["İA"] = m.ia_alis_quantity / T * 100
r["İA kamu"] = m.iaKamu / 2 / T * 100
r["İA özel"] = m.iaOzel / 2 / T * 100
r["GÖP"] = m.gop_matchedBids / T * 100
r["GİP"] = m.gip_clearingQuantityAsk / T * 100
r["DGP"] = m.dgp_mwh / T * 100
r["yenilenebilir"] = m.ren / m.uretim_total * 100
r["hidro"] = (m.uretim_dammedHydro + m.uretim_river) / m.uretim_total * 100
p = j.ptf_usd
r["PTF"] = p.resample("MS").mean(); r["oynaklık"] = p.resample("MS").std(); r["CV"] = r["oynaklık"] / r["PTF"]
r["günlük makas"] = p.resample("D").agg(lambda x: x.max() - x.min()).resample("MS").mean()
r["yil"], r["ay"] = r.index.year, r.index.month
yillik = j.groupby("yil")[["tuketim_mwh", "ia_alis_quantity", "iaKamu", "iaOzel", "gop_matchedBids", "gip_clearingQuantityAsk", "dgp_mwh"]].sum()
Y = pd.DataFrame({"İA": yillik.ia_alis_quantity, "İA kamu": yillik.iaKamu / 2, "İA özel": yillik.iaOzel / 2,
                  "GÖP": yillik.gop_matchedBids, "GİP": yillik.gip_clearingQuantityAsk, "DGP": yillik.dgp_mwh}).div(yillik.tuketim_mwh, axis=0) * 100
display(Y.round(1))
r.round(1).to_csv(ISLN / "h03_aylik_ozet.csv", encoding="utf-8-sig")

In [ ]:
# Grafik 1: yıllara göre piyasa hacimleri
fig, ax = plt.subplots(figsize=(10, 5.6))
pz = ["GÖP", "İA kamu", "İA özel", "GİP", "DGP"]
w = 0.2; x = np.arange(len(pz))
for i, y in enumerate(Y.index):
    v = Y.loc[y, pz].values
    b = ax.bar(x + (i - 1.5) * w, v, w, color=YIL_RENK[y], edgecolor=NV, lw=.8, label=str(y) + (" (Oca-Eki)" if y == 2026 else ""))
    if y in (2023, 2026):
        for xi, vi in zip(x, v): ax.text(xi + (i - 1.5) * w, vi + 1.5, f"{vi:.0f}", ha="center", fontsize=9, color=NV)
ax.axhline(100, color="#d11", ls="--", lw=1.5); ax.text(4.45, 102, "tüketim = %100", color="#d11", ha="right", fontsize=9.5)
ax.set_xticks(x, ["GÖP\n(eşleşen)", "İA kamu", "İA özel", "GİP", "Dengeleme\n(YAL+YAT)"])
ax.set_ylabel("Tüketime oran (%)"); ax.set_ylim(0, 132); ax.legend(frameon=False, ncol=4, loc="upper left")
kaydet(fig, "h03_yolculuk")

In [ ]:
# Grafik 2: aylık seyir
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.axvspan(pd.Timestamp("2026-03-01"), pd.Timestamp("2026-06-30"), color="#ffd400", alpha=.22, lw=0)
ax.text(pd.Timestamp("2026-03-05"), 183, "2026 baharı:\nPTF çöktü", fontsize=9.5, color=NV)
ax.plot(r.index, r["İA özel"], color=OR, lw=2.6, label="İA özel")
ax.plot(r.index, r["İA kamu"], color=G3, lw=2.2, label="İA kamu")
ax.plot(r.index, r["GÖP"], color=NV, lw=2.6, label="GÖP eşleşen")
ax.plot(r.index, r["DGP"], color=G2, lw=1.8, ls="--", label="Dengeleme")
ax.axhline(100, color="#d11", ls=":", lw=1.3)
ax.set_ylabel("Tüketime oran (%)"); ax.set_ylim(0, 200); ax.legend(frameon=False, ncol=4, loc="upper left")
kaydet(fig, "h03_aylik")

## 2) Saat bazında: öğle mi akşam mı daha çok GÖP'e kalıyor?

In [ ]:
g = j.groupby(["yil", "saat"]).sum(numeric_only=True)
hg = (g.gop_matchedBids / g.tuketim_mwh * 100).unstack(0)
hi = (g.ia_alis_quantity / g.tuketim_mwh * 100).unstack(0)
display(pd.concat({"GÖP": hg, "İA": hi}, axis=1).iloc[::3].round(1))
fig, axs = plt.subplots(1, 2, figsize=(10, 4.6), sharex=True)
for y in [2023, 2024, 2025, 2026]:
    axs[0].plot(hg.index, hg[y], color=YIL_RENK[y], lw=2.6 if y == 2026 else 1.8, label=str(y))
    axs[1].plot(hi.index, hi[y], color=YIL_RENK[y], lw=2.6 if y == 2026 else 1.8)
axs[0].set_title("GÖP eşleşen / tüketim (%)", color=NV, fontsize=11); axs[1].set_title("İA alış / tüketim (%)", color=NV, fontsize=11)
for a in axs: a.set_xticks(range(0, 24, 3)); a.set_xlabel("Saat")
axs[0].legend(frameon=False, ncol=2)
kaydet(fig, "h03_saatlik")

## 3) Yenilenebilir arttıkça GÖP büyüyor mu? (YEKDEM vekili)
YEKDEM UEVM servisi boş döndü. Vekil: rüzgâr + güneş + akarsu + jeotermal + biyokütle payı. Yıl ve ay sabit etkili, HAC standart hata.

In [ ]:
r2 = r.rename(columns={"İA": "ia", "İA özel": "ozel", "İA kamu": "kamu", "GÖP": "gop", "DGP": "dgp", "yenilenebilir": "ren", "günlük makas": "makas"})
bahar = pd.to_datetime(["2026-03-01", "2026-04-01", "2026-05-01", "2026-06-01"])
def model(y, x, d):
    f = smf.ols(f"{y} ~ {x} + C(yil) + C(ay)", d).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    return f.params[x], f.pvalues[x]
SON_TEST = []
for y in ["gop", "ia", "ozel", "kamu"]:
    for x in ["ren", "hidro", "PTF", "CV"]:
        b1, p1 = model(y, x, r2); b2, p2 = model(y, x, r2.drop(bahar))
        SON_TEST.append([y, x, b1, p1, b2, p2])
TEST = pd.DataFrame(SON_TEST, columns=["oran", "değişken", "eğim (tümü)", "p (tümü)", "eğim (bahar 2026 hariç)", "p (hariç)"])
display(TEST.round(3)); TEST.to_csv(ISLN / "h03_testler.csv", index=False, encoding="utf-8-sig")

## 4-5) Oynaklık ve fiyat seviyesiyle ilişki

In [ ]:
print(j.groupby("yil").ptf_usd.agg(["mean", "std"]).round(1))
fig, ax = plt.subplots(figsize=(10, 5.6))
for y in [2023, 2024, 2025, 2026]:
    d = r[r.yil == y]
    ax.scatter(d["PTF"], d["İA"], s=70, color=YIL_RENK[y], edgecolor=NV, lw=.8, label=str(y), zorder=3)
for dt, lab in [("2026-05-01", "May 2026"), ("2026-06-01", "Haz 2026"), ("2023-01-01", "Oca 2023")]:
    ax.annotate(lab, (r.loc[dt, "PTF"], r.loc[dt, "İA"]), xytext=(8, -4), textcoords="offset points", fontsize=9.5, color=NV)
ax.set_xlabel("Aylık ortalama PTF (USD/MWh)"); ax.set_ylabel("İA alış / tüketim (%)"); ax.legend(frameon=False)
kaydet(fig, "h03_ptf_ia")

In [ ]:
# Grafik: spot fiyata açık kalmanın riski
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.bar(r.index, r["günlük makas"], width=25, color=[YIL_RENK[y] for y in r.yil], edgecolor=NV, lw=.6)
ax.plot(r.index, r["PTF"], color=NV, lw=2.4, label="Aylık ortalama PTF")
ax.set_ylabel("USD/MWh")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=G2, label="Günlük en yüksek − en düşük PTF (ay ortalaması)"), plt.Line2D([], [], color=NV, lw=2.4, label="Aylık ortalama PTF")], frameon=False, loc="upper right")
kaydet(fig, "h03_oynaklik")

## 6) Piyasalar arası fiyat farkları

In [ ]:
j["d_gip"], j["d_smf"] = j.gip_usd - j.ptf_usd, j.smf_usd - j.ptf_usd
F = j.groupby("yil").agg(PTF=("ptf_usd", "mean"), GİP=("gip_usd", "mean"), SMF=("smf_usd", "mean"),
    gip_fark_ort=("d_gip", "mean"), gip_fark_mutlak=("d_gip", lambda x: x.abs().mean()),
    smf_fark_ort=("d_smf", "mean"), smf_fark_mutlak=("d_smf", lambda x: x.abs().mean()))
YON = j.groupby(["yil", "sistem_yonu"]).d_smf.mean().unstack()
display(F.round(1)); display(YON.round(1))
fig, axs = plt.subplots(1, 2, figsize=(10, 4.6))
yy = F.index.astype(str)
axs[0].bar(np.arange(4) - .2, F.gip_fark_mutlak, .4, color=G2, edgecolor=NV, label="GİP")
axs[0].bar(np.arange(4) + .2, F.smf_fark_mutlak, .4, color=OR, edgecolor=NV, label="SMF")
axs[0].set_xticks(range(4), yy); axs[0].set_title("PTF'den ortalama mutlak sapma\n(USD/MWh, saatlik)", color=NV, fontsize=10.5); axs[0].legend(frameon=False)
axs[1].bar(np.arange(4) - .2, YON["acik"], .4, color=OR, edgecolor=NV, label="Enerji açığı saatleri")
axs[1].bar(np.arange(4) + .2, YON["fazla"], .4, color=G3, edgecolor=NV, label="Enerji fazlası saatleri")
axs[1].axhline(0, color=NV, lw=1); axs[1].set_xticks(range(4), yy)
axs[1].set_title("SMF − PTF, sistem yönüne göre\n(USD/MWh, ortalama)", color=NV, fontsize=10.5); axs[1].legend(frameon=False, fontsize=9)
kaydet(fig, "h03_fiyatlar")

## 7) Özet

In [ ]:
oz = {"yillik_oran": Y.round(1).to_dict(), "saatlik_gop_2026": hg[2026].round(1).to_dict(),
      "fiyat_fark": F.round(2).to_dict(), "smf_yon": YON.round(1).to_dict(),
      "bahar_2026": r.loc["2026-03":"2026-06", ["İA", "İA özel", "GÖP", "DGP", "PTF", "CV"]].round(2).set_axis(["Mar", "Nis", "May", "Haz"]).to_dict()}
json.dump(oz, open(OUT / "ozet_h03.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
print("Bitti. Grafikler:", OUT)